In [2]:
import gymnasium as gym
import numpy as np

In [4]:
env = gym.make("CartPole-v1")

LEARNING_RATE = 0.1
DISCOUNT = 0.95
EPISODES = 20000

epsilon = 1
START_EPSILON_DECAYING = 1
END_EPSILON_DECAYING = EPISODES // 2
epsilon_decay = epsilon / (END_EPSILON_DECAYING - START_EPSILON_DECAYING)



In [5]:
# Discretization
DISCRETE_OS_SIZE = [20] * len(env.observation_space.high)

high = env.observation_space.high
low = env.observation_space.low

In [6]:
# Fix infinite bounds
high[1] = 5
high[3] = 5
low[1] = -5
low[3] = -5


In [7]:
discrete_os_win_size = (high - low) / DISCRETE_OS_SIZE

# Q-table
q_table = np.random.uniform(low=-2, high=0,
                            size=(DISCRETE_OS_SIZE + [env.action_space.n]))






In [8]:
def get_discrete_state(state):
    discrete_state = (state - low) / discrete_os_win_size
    return tuple(discrete_state.astype(int))

In [9]:
for episode in range(EPISODES):

    state, _ = env.reset()
    discrete_state = get_discrete_state(state)

    done = False

    while not done:

        if np.random.random() > epsilon:
            action = np.argmax(q_table[discrete_state])
        else:
            action = env.action_space.sample()

        new_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        new_discrete_state = get_discrete_state(new_state)

        if not done:
            max_future_q = np.max(q_table[new_discrete_state])
            current_q = q_table[discrete_state + (action,)]

            new_q = (1 - LEARNING_RATE) * current_q + LEARNING_RATE * (
                reward + DISCOUNT * max_future_q
            )

            q_table[discrete_state + (action,)] = new_q

        discrete_state = new_discrete_state

    if END_EPSILON_DECAYING >= episode >= START_EPSILON_DECAYING:
        epsilon -= epsilon_decay



In [10]:
env.close()

### After Training

In [11]:
env = gym.make("CartPole-v1", render_mode="human")

state, _ = env.reset()

done = False
while not done:
    action = np.argmax(q_table[get_discrete_state(state)])
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

env.close()

/home/sachchida/anaconda3/envs/rl/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
